## Guidelines for Prompting
In this lesson, we'll practice two prompting principles and their related tactics in order to write effective prompts for large language models.

In [1]:
# load api key and relevant libraries

import openai
import os


from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
model = "openai/gpt-4.1"

client = openai.OpenAI(
    base_url=endpoint,
    api_key=token,
)



In [12]:
# helper function to get a completion from OpenAI API
def get_completion(prompt, model=model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content


### Prompting Principles
**Principle 1: Write clear and specific instructions**    
**Principle 2: Give the model time to “think”**

#### Principle 1 :  Tactics
**Tactic 1: Use delimiters to clearly indicate distinct parts of the input**
- Delimiters can be anything like: ```, """, < >, <tag> </tag>, :

In [26]:
text = f"""
You should express what you want a model to do by \
providing instructions that are as clear and \
specific as you can possibly make them. \
This will guide the model towards the desired output, \
and reduce the chances of receiving irrelevant \
or incorrect responses. Don't confuse writing a \
clear prompt with writing a short prompt. \
In many cases, longer prompts provide more clarity \
and context for the model, which can lead to \
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

Clear and specific instructions, even if lengthy, help guide a model to produce accurate and relevant responses.


**Tactic 2: Ask for a structured output**
- JSON, HTML

In [ ]:
prompt = f"""
Generate a list of three famous  book titles along  with their authors and genres. 
Provide them in JSON format with the following keys:  book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

[
  {
    "book_id": 1,
    "title": "To Kill a Mockingbird",
    "author": "Harper Lee",
    "genre": "Fiction"
  },
  {
    "book_id": 2,
    "title": "1984",
    "author": "George Orwell",
    "genre": "Dystopian"
  },
  {
    "book_id": 3,
    "title": "The Great Gatsby",
    "author": "F. Scott Fitzgerald",
    "genre": "Classic"
  }
]


**Tactic 3: Ask the model to check whether conditions are satisfied**

In [ ]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some  water boiling. While that's happening, grab a cup and put a tea bag in it. Once the water is hot enough, just pour it over the tea bag. Let it sit for a bit so the tea can steep. After a few minutes, take out the tea bag. If you  like, you can add some sugar or milk to taste. And that's it! You've got yourself a delicious  cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Boil some water.
Step 2 - Grab a cup and put a tea bag in it.
Step 3 - Pour the hot water over the tea bag.
Step 4 - Let the tea steep for a few minutes.
Step 5 - Remove the tea bag.
Step 6 - Add sugar or milk to taste, if desired.


In [ ]:
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \
walk in the park. The flowers are blooming, and the \
trees are swaying gently in the breeze. People \
are out and about, enjoying the lovely weather. \
Some are having picnics, while others are playing \
games or simply relaxing on the grass. It's a \
perfect day to spend time outdoors and appreciate the \
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


#### Principle 2: Give the model time to “think”
**Tactic 1: Specify the steps required to complete a task**

In [27]:
text = f"""
In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop well. As they climbed, singing joyfully, misfortune struck—Jack tripped on a stone and tumbled  down the hill, with Jill following suit. Though slightly battered, the pair returned home to comforting embraces. Despite the mishap, their adventurous spirits remained undimmed, and they  continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
1. Jack and Jill, siblings from a charming village, went to fetch water from a hilltop well, but after tumbling down the hill, they returned home undeterred and continued their adventures.

2. Jack et Jill, frères et sœurs d’un charmant village, sont allés chercher de l’eau à un puits au sommet d’une colline, mais après avoir dévalé la pente, ils sont rentrés chez eux sans se décourager et ont poursuivi leurs aventures.

3. Jack, Jill

4. 
```json
{
  "french_summary": "Jack et Jill, frères et sœurs d’un charmant village, sont allés chercher de l’eau à un puits au sommet d’une colline, mais après avoir dévalé la pente, ils sont rentrés chez eux sans se décourager et ont poursuivi leurs aventures.",
  "num_names": 2
}
```


Ask for output in a specified format

In [28]:
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Summary: Siblings Jack and Jill went to fetch water from a hilltop well, but after Jack fell and Jill followed, they returned home safely and kept their adventurous spirit.

Translation: Les frères et sœurs Jack et Jill sont allés chercher de l'eau à un puits au sommet de la colline, mais après que Jack soit tombé et que Jill l'ait suivi, ils sont rentrés chez eux sains et saufs et ont gardé leur esprit d'aventure.

Names: Jack, Jill

Output JSON: {
  "french_summary": "Les frères et sœurs Jack et Jill sont allés chercher de l'eau à un puits au sommet de la colline, mais après que Jack soit tombé et que Jill l'ait suivi, ils sont rentrés chez eux sains et saufs et ont gardé leur esprit d'aventure.",
  "num_names": 2
}


**Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion**

In [13]:
# example showing -  sometimes model give wrong response when not asked to be detailed or asked in brief (this example is easy so it gives correct response some times and sometimes not) 
prompt = f"""
Determine if the student's solution is correct or not (just tell correct or not) .

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)

Correct.


**Note that the student's solution is actually not correct.        
We can fix this by instructing the model to work out its own solution first.**

In [35]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution \
and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

```
Let x be the number of square feet.

Land cost: $100 per square foot
Land cost = 100x

Solar panel cost: $250 per square foot
Solar panel cost = 250x

Maintenance cost: $100,000 flat per year + $10 per square foot
Maintenance cost = 100,000 + 10x

Total cost for first year:
= Land cost + Solar panel cost + Maintenance cost
= 100x + 250x + (100,000 + 10x)
= (100x + 250x + 10x) + 100,000
= (360x) + 100,000

So, the total cost for the first year as a function of x is:
Total cost = 360x + 100,000
```
Is the student's solution the same as actual solution just calculated:
```
No. The student used $100 per square foot for maintenance instead of $10 per square foot, resulting in 450x instead of 360x.
```
Student grade:
```
Incorrect
```


### Model Limitations: Hallucinations
- Boie is a real company, the product name is not real.

In [37]:
prompt = f"""
List five peer-reviewed scientific sources proving that Atlantis existed, including authors, journal names, years, and DOIs.
"""
# prompt = f"""
# Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
# """
response = get_completion(prompt)
print(response)

I'm sorry, but there are **no peer-reviewed scientific sources that prove Atlantis existed**. Atlantis is widely regarded by historians, archaeologists, and scientists as a myth or allegory first described by Plato in his dialogues "Timaeus" and "Critias." While there are many speculative and pseudoscientific works about Atlantis, **no credible, peer-reviewed scientific journal has published evidence proving its existence**.

If you are interested in scholarly discussions about the myth, its origins, or its impact on culture and archaeology, I can provide references to academic analyses of the Atlantis story. Let me know if you would like such sources.
